# Zadanie 1: grafik pracy na SOR
Zastosowanie Biznesowe Sztucznej Inteligencji, Vladyslav Rybak, album 120750, grupa 2

Ustalam obsadę 9 zmian (3 dni po 3 zmiany) tak, żeby koszt był jak najniższy. Model zrobiłem w OR-Tools (CP-SAT).

In [1]:
!pip install ortools pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 57.6 MB/s  0:00:00


Najpierw importy i dane

In [2]:
import os
import pandas as pd
from ortools.sat.python import cp_model

if not os.path.exists("L1_zad.csv"):
    from google.colab import files
    files.upload()

dane = pd.read_csv("L1_zad.csv")
dane

,pracownik,rola,stawka_h,max_zmian,Pon_Rano,Pon_Popoludnie,Pon_Noc,Wt_Rano,Wt_Popoludnie,Wt_Noc,Sr_Rano,Sr_Popoludnie,Sr_Noc
0,Dr_Kowalski,Senior,140,2,1,1,0,1,1,0,1,0,0
1,Dr_Nowakowska,Senior,135,2,0,1,1,0,1,1,1,1,0
2,Rat_Wisniewski,Senior,110,3,1,1,1,1,1,1,0,1,1
3,Rat_Wojcik,Senior,105,3,1,0,1,1,1,0,1,1,1
4,Piel_Kaminska,Senior,95,3,1,1,1,1,1,1,1,1,1
5,Piel_Lewandowska,Junior,75,3,1,1,0,1,1,1,1,1,0
6,Piel_Zielinska,Junior,75,3,0,1,1,1,1,0,1,1,1
7,Rat_Szymanski,Junior,70,3,1,1,1,0,1,1,1,0,1
8,Rat_Wozniak,Junior,70,3,1,0,1,1,1,1,0,1,1
9,Piel_Dabrowska,Junior,68,3,1,1,1,1,0,1,1,1,0


In [3]:
dni = ["Pon", "Wt", "Sr"]
pory = ["Rano", "Popoludnie", "Noc"]

zmiany = []
for dzien in dni:
    for pora in pory:
        zmiany.append(dzien + "_" + pora)

godziny_zmiany = 8
liczba_pracownikow = len(dane)

Zmienna `x[p, s]` jest równa 1, gdy pracownik `p` pracuje na zmianie `s`. Potem ograniczenia z treści zadania: po 2 osoby na zmianie, przynajmniej 1 senior, limit `max_zmian` i zakaz zmiany porannej po nocnej.

In [4]:
model = cp_model.CpModel()

x = {}
for p in range(liczba_pracownikow):
    for s in zmiany:
        x[p, s] = model.NewBoolVar("x_" + str(p) + "_" + s)
        # niedostępny -> od razu 0
        if dane.loc[p, s] == 0:
            model.Add(x[p, s] == 0)

In [5]:
for s in zmiany:
    model.Add(sum(x[p, s] for p in range(liczba_pracownikow)) == 2)

In [6]:
for s in zmiany:
    seniorzy_na_zmianie = []
    for p in range(liczba_pracownikow):
        if dane.loc[p, "rola"] == "Senior":
            seniorzy_na_zmianie.append(x[p, s])
    model.Add(sum(seniorzy_na_zmianie) >= 1)

In [7]:
for p in range(liczba_pracownikow):
    model.Add(sum(x[p, s] for s in zmiany) <= dane.loc[p, "max_zmian"])

In [8]:
# rano po nocy jest zabronione
for p in range(liczba_pracownikow):
    for i in range(len(dni) - 1):
        model.Add(x[p, dni[i] + "_Noc"] + x[p, dni[i + 1] + "_Rano"] <= 1)

Dodałem jeszcze jedno ograniczenie od siebie: maksymalnie 1 zmiana dziennie. Bez tego ktoś mógłby mieć rano i noc tego samego dnia, czyli tylko 8 godzin przerwy, a w zadaniu po nocy wymagane jest 11. Razem z zakazem rano po nocy wyklucza to też dwie zmiany pod rząd.

In [9]:
zalozenie_max_1_zmiana_dziennie = True

if zalozenie_max_1_zmiana_dziennie:
    for p in range(liczba_pracownikow):
        for dzien in dni:
            model.Add(x[p, dzien + "_Rano"] + x[p, dzien + "_Popoludnie"]
                      + x[p, dzien + "_Noc"] <= 1)

Drugie ograniczenie też od siebie i opcjonalne: każdy dostaje co najmniej 1 zmianę. W optymalnym grafiku 6 z 12 osób w ogóle nie pracuje, bo 6 najtańszych pokrywa wszystkie 18 miejsc. Ta opcja dzieli pracę sprawiedliwiej ale wychodzi drożej. Zostawiłem ją wyłączoną, bo zadanie wymaga najtańszego grafiku.

In [10]:
zalozenie_min_1_zmiana = False

if zalozenie_min_1_zmiana:
    for p in range(liczba_pracownikow):
        model.Add(sum(x[p, s] for s in zmiany) >= 1)

Funkcja celu to suma `stawka_h * 8` po wszystkich przydzielonych zmianach. Potem uruchamiam solver.

In [11]:
koszt_calkowity = 0
for p in range(liczba_pracownikow):
    for s in zmiany:
        koszt_zmiany = dane.loc[p, "stawka_h"] * godziny_zmiany * x[p, s]
        koszt_calkowity = koszt_calkowity + koszt_zmiany

model.Minimize(koszt_calkowity)

In [12]:
solver = cp_model.CpSolver()
solver.parameters.num_workers = 1
solver.parameters.random_seed = 42
status = solver.Solve(model)

print("Status:", solver.StatusName(status))
koszt = round(solver.ObjectiveValue())
print("Całkowity koszt grafiku:", f"{koszt:,}".replace(",", " "), "PLN")

Status: OPTIMAL
Całkowity koszt grafiku: 12 192 PLN


Na koniec wypisuję grafik (kto na której zmianie), liczbę zmian każdego pracownika i sprawdzam, czy wszystko się zgadza.

In [13]:
wiersze = []
for s in zmiany:
    osoby = []
    for p in range(liczba_pracownikow):
        if solver.Value(x[p, s]) == 1:
            osoby.append(dane.loc[p, "pracownik"] + " (" + dane.loc[p, "rola"] + ", " + str(dane.loc[p, "stawka_h"]) + " PLN/h)")
    wiersze.append({"Zmiana": s, "Pracownik 1": osoby[0], "Pracownik 2": osoby[1]})

grafik = pd.DataFrame(wiersze)
grafik

,Zmiana,Pracownik 1,Pracownik 2
0,Pon_Rano,"Rat_Wojcik (Senior, 105 PLN/h)","Piel_Dabrowska (Junior, 68 PLN/h)"
1,Pon_Popoludnie,"Piel_Kaminska (Senior, 95 PLN/h)","Piel_Kozlowska (Junior, 65 PLN/h)"
2,Pon_Noc,"Rat_Wisniewski (Senior, 110 PLN/h)","Rat_Jankowski (Junior, 65 PLN/h)"
3,Wt_Rano,"Piel_Kaminska (Senior, 95 PLN/h)","Piel_Dabrowska (Junior, 68 PLN/h)"
4,Wt_Popoludnie,"Rat_Wojcik (Senior, 105 PLN/h)","Piel_Kozlowska (Junior, 65 PLN/h)"
5,Wt_Noc,"Rat_Wisniewski (Senior, 110 PLN/h)","Rat_Jankowski (Junior, 65 PLN/h)"
6,Sr_Rano,"Piel_Kaminska (Senior, 95 PLN/h)","Piel_Dabrowska (Junior, 68 PLN/h)"
7,Sr_Popoludnie,"Rat_Wisniewski (Senior, 110 PLN/h)","Rat_Jankowski (Junior, 65 PLN/h)"
8,Sr_Noc,"Rat_Wojcik (Senior, 105 PLN/h)","Piel_Kozlowska (Junior, 65 PLN/h)"


In [14]:
liczby_zmian = []
for p in range(liczba_pracownikow):
    ile = 0
    for s in zmiany:
        ile = ile + solver.Value(x[p, s])
    liczby_zmian.append(ile)

podsumowanie = pd.DataFrame({"Pracownik": dane["pracownik"], "Rola": dane["rola"],
                             "Stawka [PLN/h]": dane["stawka_h"], "Liczba zmian": liczby_zmian})
podsumowanie

,Pracownik,Rola,Stawka [PLN/h],Liczba zmian
0,Dr_Kowalski,Senior,140,0
1,Dr_Nowakowska,Senior,135,0
2,Rat_Wisniewski,Senior,110,3
3,Rat_Wojcik,Senior,105,3
4,Piel_Kaminska,Senior,95,3
5,Piel_Lewandowska,Junior,75,0
6,Piel_Zielinska,Junior,75,0
7,Rat_Szymanski,Junior,70,0
8,Rat_Wozniak,Junior,70,0
9,Piel_Dabrowska,Junior,68,3


In [15]:

assert status == cp_model.OPTIMAL, "brak optimum"
for s in zmiany:
    liczba_osob = 0
    liczba_seniorow = 0
    for p in range(liczba_pracownikow):
        liczba_osob = liczba_osob + solver.Value(x[p, s])
        if dane.loc[p, "rola"] == "Senior":
            liczba_seniorow = liczba_seniorow + solver.Value(x[p, s])
    assert liczba_osob == 2, "zła obsada: " + s
    assert liczba_seniorow >= 1, "brak seniora: " + s
for p in range(liczba_pracownikow):
    assert liczby_zmian[p] <= dane.loc[p, "max_zmian"], "za dużo zmian"
    for i in range(len(dni) - 1):
        noc = solver.Value(x[p, dni[i] + "_Noc"])
        rano = solver.Value(x[p, dni[i + 1] + "_Rano"])
        assert noc + rano <= 1, "rano po nocy: " + dane.loc[p, "pracownik"]
print("Wszystko się zgadza.")

Wszystko się zgadza.
